# Week 5 · Day 3 — LangGraph: Stateful, Multi-Step & Cyclical Agent Workflows
### Groq API edition

**Note on API provider:** built with `langchain-groq` (Groq API) via `.env`, consistent
with Days 1–2. LangGraph itself is model-agnostic — it wraps whatever chat model you give
it, so nothing about the graph logic below is Groq-specific.

**Setup**

```bash
pip install langgraph langchain-groq langchain-core python-dotenv pydantic
```

`.env` (same file as Day 2):
```
GROQ_API_KEY=gsk_your_key_here
```

**Scenario for this notebook:** a small research-assistant workflow that helps recommend
a product to a budget-conscious client. It looks up prices (reusing the Day 2
`lookup_price`-style tool), drafts a recommendation, critiques its own draft, loops back
to revise if the critique isn't good enough, pauses for human approval before "publishing"
the recommendation (the risky, external-facing action), and persists its state so a
paused run can be resumed later.


In [1]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

from langchain_groq import ChatGroq

MODEL = "llama-3.3-70b-versatile"
llm = ChatGroq(model=MODEL, api_key=os.environ.get("GROQ_API_KEY"), temperature=0)


---
## Task 1 — Graph Concepts & State Design

### Core building blocks

- **`StateGraph`** — the graph builder. You define a state schema, add nodes and edges to
  it, then `.compile()` it into a runnable graph.
- **Node** — a plain Python function `(state) -> dict`. It receives the current state and
  returns a partial update, which LangGraph merges back into the shared state. A node is
  the graph equivalent of one step in Day 1's loop body (one Reason/Act/Observe pass), but
  as an explicit, individually addressable unit instead of an iteration of a `for` loop.
- **Edge** — a fixed transition: "after node A, always go to node B." Defined with
  `add_edge(A, B)`.
- **Conditional edge** — a transition chosen at run time by a routing function that
  inspects the current state and returns the name of the next node. Defined with
  `add_conditional_edges(A, routing_fn, {...})`. This is what makes branching and
  loop-back possible — a routing function can send execution *back* to a node already
  visited, which is exactly how a self-correction loop is built.
- **State** — a shared, typed object (here a `TypedDict`) that every node reads from and
  writes back to. It's the explicit, structured version of Day 1's `messages` list and
  Day 2's `agent_scratchpad` — instead of one growing transcript, you get named fields
  you control precisely.

### State schema for this workflow

```python
class ResearchState(TypedDict):
    question: str            # the client's request
    target_products: list    # products identified for lookup
    retrieved_data: str      # raw price/spec data pulled by the retrieve node
    draft: str                # current draft recommendation
    critique: str             # feedback from the critique node
    quality_score: float      # 0-1 self-assessed quality of the draft
    revision_count: int       # how many times we've looped back to generate
    max_revisions: int        # cap to guarantee termination
    approved: bool | None      # human-in-the-loop decision, None until reviewed
    final_answer: str          # the published recommendation
```

### Graph diagram (planned before coding)

```
                 ┌─────────┐
        START ──▶│  plan   │
                 └────┬────┘
                      ▼
                 ┌─────────┐
                 │retrieve │
                 └────┬────┘
                      ▼
              ┌──────────────┐
        ┌────▶│   generate   │
        │     └──────┬───────┘
        │            ▼
        │      ┌───────────┐
        │      │ critique  │
        │      └─────┬─────┘
        │            │  conditional edge
        │   score < threshold        score >= threshold
        │   & revisions < max        OR revisions >= max
        └────────(loop back)               │
                                            ▼
                                  ┌───────────────────┐
                                  │  publish_gate      │  <- interrupt_before:
                                  │ (human approval)   │     pauses here
                                  └─────────┬──────────┘
                                approved?  /   \ rejected?
                                          ▼      ▼
                                  ┌─────────┐  ┌──────┐
                                  │ publish │  │ END  │
                                  └────┬────┘  └──────┘
                                       ▼
                                      END
```


---
## Task 2 — Build a Linear Graph

First, the linear backbone: `plan -> retrieve -> generate -> format`, reusing the Day 2
product catalog as the "retrieve" data source. Conditional edges and the loop come in
Task 3 -- this section proves the plumbing works on a straight line first.


In [2]:
from typing import TypedDict, Optional, List

class ResearchState(TypedDict):
    question: str
    target_products: List[str]
    retrieved_data: str
    draft: str
    critique: str
    quality_score: float
    revision_count: int
    max_revisions: int
    approved: Optional[bool]
    final_answer: str


In [3]:
# --- Reused from Day 2: a tiny local JSON product catalog -------------------
PRODUCTS_PATH = "products.json"
products_db = {
    "laptop a": {"price_usd": 799, "specs": "8GB RAM, 256GB SSD"},
    "laptop b": {"price_usd": 1199, "specs": "16GB RAM, 512GB SSD"},
    "laptop c": {"price_usd": 549, "specs": "8GB RAM, 128GB SSD"},
}
with open(PRODUCTS_PATH, "w") as f:
    json.dump(products_db, f, indent=2)


def lookup_price(product_name: str) -> str:
    with open(PRODUCTS_PATH) as f:
        db = json.load(f)
    key = product_name.strip().lower()
    if key not in db:
        return f"{product_name}: not found in catalog."
    e = db[key]
    return f"{product_name}: ${e['price_usd']} ({e['specs']})"


In [4]:
# --- Node functions ----------------------------------------------------------

def plan_node(state: ResearchState) -> dict:
    """Reads the question and decides which catalog products to look up."""
    response = llm.invoke(
        f"A client asked: '{state['question']}'. Which product names from this "
        f"catalog should be looked up to answer it: {list(products_db.keys())}? "
        f"Reply with a comma-separated list of product names only, nothing else."
    )
    products = [p.strip() for p in response.content.split(",") if p.strip()]
    print(f"[PLAN] target_products = {products}")
    return {"target_products": products}


def retrieve_node(state: ResearchState) -> dict:
    """Looks up each planned product in the catalog (the 'touches external data' step)."""
    results = [lookup_price(p) for p in state["target_products"]]
    data = "\n".join(results)
    print(f"[RETRIEVE] {data}")
    return {"retrieved_data": data}


def generate_node(state: ResearchState) -> dict:
    """Drafts a recommendation using the retrieved data (and any prior critique)."""
    guidance = f"\n\nPrevious critique to address: {state['critique']}" if state.get("critique") else ""
    response = llm.invoke(
        f"Client question: {state['question']}\n"
        f"Product data:\n{state['retrieved_data']}{guidance}\n\n"
        f"Write a short (2-3 sentence) recommendation."
    )
    print(f"[GENERATE] draft = {response.content[:100]}...")
    return {"draft": response.content, "revision_count": state.get("revision_count", 0)}


def format_node(state: ResearchState) -> dict:
    """Formats the draft into the final answer (end of the linear-only version)."""
    final = f"Recommendation:\n{state['draft']}"
    print(f"[FORMAT] final_answer set")
    return {"final_answer": final}


In [5]:
from langgraph.graph import StateGraph, START, END

# --- Linear graph: plan -> retrieve -> generate -> format -------------------
linear_builder = StateGraph(ResearchState)
linear_builder.add_node("plan", plan_node)
linear_builder.add_node("retrieve", retrieve_node)
linear_builder.add_node("generate", generate_node)
linear_builder.add_node("format", format_node)

linear_builder.add_edge(START, "plan")
linear_builder.add_edge("plan", "retrieve")
linear_builder.add_edge("retrieve", "generate")
linear_builder.add_edge("generate", "format")
linear_builder.add_edge("format", END)

linear_graph = linear_builder.compile()


In [6]:
# Run it and print state after each node to verify updates.
initial_state = {
    "question": "Which laptop should I recommend to a budget-conscious client?",
    "target_products": [],
    "retrieved_data": "",
    "draft": "",
    "critique": "",
    "quality_score": 0.0,
    "revision_count": 0,
    "max_revisions": 2,
    "approved": None,
    "final_answer": "",
}

for step in linear_graph.stream(initial_state):
    node_name = list(step.keys())[0]
    print(f"\n--- after node: {node_name} ---")
    print(step[node_name])


[PLAN] target_products = ['laptop a', 'laptop b', 'laptop c']

--- after node: plan ---
{'target_products': ['laptop a', 'laptop b', 'laptop c']}
[RETRIEVE] laptop a: $799 (8GB RAM, 256GB SSD)
laptop b: $1199 (16GB RAM, 512GB SSD)
laptop c: $549 (8GB RAM, 128GB SSD)

--- after node: retrieve ---
{'retrieved_data': 'laptop a: $799 (8GB RAM, 256GB SSD)\nlaptop b: $1199 (16GB RAM, 512GB SSD)\nlaptop c: $549 (8GB RAM, 128GB SSD)'}
[GENERATE] draft = For a budget-conscious client, I would recommend Laptop C, which offers a great balance of features ...

--- after node: generate ---
{'draft': 'For a budget-conscious client, I would recommend Laptop C, which offers a great balance of features and affordability at $549. With 8GB RAM and a 128GB SSD, it provides sufficient performance for everyday tasks while being the most cost-effective option. This choice allows your client to stay within their budget while still getting a reliable laptop for basic use.', 'revision_count': 0}
[FORMAT] final_

---
## Task 3 — Add Conditional Edges & Cycles

Replace `format` with a `critique` node that scores the draft. A conditional edge routes
back to `generate` (a self-correction loop) if the score is below a threshold and we
haven't hit `max_revisions` yet; otherwise it moves forward.


In [7]:
def critique_node(state: ResearchState) -> dict:
    """Scores the current draft and returns feedback + an updated revision_count."""
    response = llm.invoke(
        f"Rate this recommendation from 0.0 to 1.0 for clarity and whether it "
        f"actually names a specific product and price. Recommendation:\n"
        f"{state['draft']}\n\n"
        f"Reply in the exact format: SCORE: <number>\nFEEDBACK: <one sentence>"
    )
    text = response.content
    try:
        score_line = [l for l in text.splitlines() if l.upper().startswith("SCORE")][0]
        score = float(score_line.split(":")[1].strip())
    except Exception:
        score = 0.5  # fallback if the model didn't follow the format
    feedback_lines = [l for l in text.splitlines() if l.upper().startswith("FEEDBACK")]
    feedback = feedback_lines[0].split(":", 1)[1].strip() if feedback_lines else ""

    revision_count = state.get("revision_count", 0)
    print(f"[CRITIQUE] pass {revision_count}: score={score}, feedback={feedback!r}")
    return {"quality_score": score, "critique": feedback, "revision_count": revision_count + 1}


def route_after_critique(state: ResearchState) -> str:
    """Conditional edge: loop back to generate, or move on to the approval gate."""
    below_threshold = state["quality_score"] < 0.7
    retries_left = state["revision_count"] < state["max_revisions"]
    if below_threshold and retries_left:
        print(f"[ROUTE] score {state['quality_score']} < 0.7 and retries left -> back to generate")
        return "generate"
    print(f"[ROUTE] score {state['quality_score']} acceptable or out of retries -> publish_gate")
    return "publish_gate"


**Why this is natural in LangGraph but awkward in a plain `AgentExecutor`:** `AgentExecutor`
models a single loop where the *only* decision point is "call another tool, or stop" — it
has no concept of named steps you can route between, so a "critique, then conditionally
go back to an earlier step" pattern would have to be smuggled into the system prompt as an
instruction ("if not good enough, try again") with no structural guarantee it's followed,
and no separate place to track how many times you've retried. In LangGraph, `generate` and
`critique` are distinct nodes and the loop-back is an explicit edge with its own routing
function and its own `revision_count` field in state — the control flow is declared in
code, not inferred by the model from a prompt.


---
## Task 4 — Human-in-the-Loop & Interrupts

Add a `publish_gate` node before the (simulated) risky action -- actually sending the
recommendation to the client. The graph is compiled with `interrupt_before=["publish_gate"]`
so it pauses there and waits for a human decision before continuing.


In [8]:
def publish_gate_node(state: ResearchState) -> dict:
    """No-op node whose only purpose is to be an interrupt point before publishing."""
    print("[PUBLISH_GATE] paused for human approval")
    return {}


def route_after_gate(state: ResearchState) -> str:
    return "publish" if state.get("approved") else "rejected_end"


def publish_node(state: ResearchState) -> dict:
    """The 'risky' action: this is where we'd actually email/send the recommendation."""
    final = f"[SENT TO CLIENT] Recommendation:\n{state['draft']}"
    print("[PUBLISH] recommendation sent")
    return {"final_answer": final}


def rejected_end_node(state: ResearchState) -> dict:
    print("[REJECTED] human declined to send the recommendation")
    return {"final_answer": "Recommendation was not sent (rejected by reviewer)."}


In [9]:
from langgraph.checkpoint.memory import MemorySaver

full_builder = StateGraph(ResearchState)
full_builder.add_node("plan", plan_node)
full_builder.add_node("retrieve", retrieve_node)
full_builder.add_node("generate", generate_node)
full_builder.add_node("critique", critique_node)
full_builder.add_node("publish_gate", publish_gate_node)
full_builder.add_node("publish", publish_node)
full_builder.add_node("rejected_end", rejected_end_node)

full_builder.add_edge(START, "plan")
full_builder.add_edge("plan", "retrieve")
full_builder.add_edge("retrieve", "generate")
full_builder.add_edge("generate", "critique")
full_builder.add_conditional_edges(
    "critique", route_after_critique, {"generate": "generate", "publish_gate": "publish_gate"}
)
full_builder.add_conditional_edges(
    "publish_gate", route_after_gate, {"publish": "publish", "rejected_end": "rejected_end"}
)
full_builder.add_edge("publish", END)
full_builder.add_edge("rejected_end", END)

# --- Task 5 checkpointer lives here too: needed for interrupt_before to work ---
checkpointer = MemorySaver()

full_graph = full_builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["publish_gate"],
)


In [10]:
# A thread_id identifies one persistent conversation/run for the checkpointer.
config = {"configurable": {"thread_id": "client-session-1"}}

initial_state = {
    "question": "Which laptop should I recommend to a budget-conscious client?",
    "target_products": [],
    "retrieved_data": "",
    "draft": "",
    "critique": "",
    "quality_score": 0.0,
    "revision_count": 0,
    "max_revisions": 2,
    "approved": None,
    "final_answer": "",
}

# Runs plan -> retrieve -> generate -> critique -> (loop or not) -> pauses
# before publish_gate because of interrupt_before.
for step in full_graph.stream(initial_state, config=config):
    print(step)

print("\nGraph is now paused. Current state:")
print(full_graph.get_state(config).values)


[PLAN] target_products = ['laptop a', 'laptop b', 'laptop c']
{'plan': {'target_products': ['laptop a', 'laptop b', 'laptop c']}}
[RETRIEVE] laptop a: $799 (8GB RAM, 256GB SSD)
laptop b: $1199 (16GB RAM, 512GB SSD)
laptop c: $549 (8GB RAM, 128GB SSD)
{'retrieve': {'retrieved_data': 'laptop a: $799 (8GB RAM, 256GB SSD)\nlaptop b: $1199 (16GB RAM, 512GB SSD)\nlaptop c: $549 (8GB RAM, 128GB SSD)'}}
[GENERATE] draft = For a budget-conscious client, I would recommend Laptop C, which offers a great balance of features ...
{'generate': {'draft': 'For a budget-conscious client, I would recommend Laptop C, which offers a great balance of features and affordability at $549. With 8GB RAM and a 128GB SSD, it provides sufficient performance for everyday tasks while being the most cost-effective option. This choice allows your client to stay within their budget while still getting a reliable laptop for basic use.', 'revision_count': 0}}
[CRITIQUE] pass 0: score=0.9, feedback='The recommendation is c

In [11]:
# --- Simulate human review ---------------------------------------------------
# In a real product this would be a UI action; here we just set `approved`
# directly on the checkpointed state and resume execution with `None` as input
# (meaning: "continue from where you paused, don't start over").

decision = True  # flip to False to see the rejection path

full_graph.update_state(config, {"approved": decision})

for step in full_graph.stream(None, config=config):
    print(step)

print("\nFinal state:")
print(full_graph.get_state(config).values["final_answer"])


[ROUTE] score 0.9 acceptable or out of retries -> publish_gate
[PUBLISH_GATE] paused for human approval
{'publish_gate': None}
[PUBLISH] recommendation sent
{'publish': {'final_answer': '[SENT TO CLIENT] Recommendation:\nFor a budget-conscious client, I would recommend Laptop C, which offers a great balance of features and affordability at $549. With 8GB RAM and a 128GB SSD, it provides sufficient performance for everyday tasks while being the most cost-effective option. This choice allows your client to stay within their budget while still getting a reliable laptop for basic use.'}}

Final state:
[SENT TO CLIENT] Recommendation:
For a budget-conscious client, I would recommend Laptop C, which offers a great balance of features and affordability at $549. With 8GB RAM and a 128GB SSD, it provides sufficient performance for everyday tasks while being the most cost-effective option. This choice allows your client to stay within their budget while still getting a reliable laptop for basic 

### When should a real product require human-in-the-loop vs. full autonomy?

Human-in-the-loop earns its cost when an action is **hard to reverse, externally visible,
or expensive if wrong** — sending a message to a real client, making a payment, deleting
data, deploying code. In those cases the downside of an unreviewed mistake outweighs the
friction of a pause. Full autonomy is reasonable when actions are **cheap, reversible, and
internal** — looking up a price, drafting text nobody sees yet, querying a read-only API —
where a mistake just costs a retry, not real-world consequences. In between, a useful
default is to gate the *last, externally-facing* step (as done here with `publish_gate`)
while leaving the research/drafting steps fully autonomous.


---
## Task 5 — Persistence & Debugging

The `MemorySaver` checkpointer from Task 4 already gives us persistence -- the paused run
above survives as long as the process is alive, keyed by `thread_id`. This section shows
resuming a *separately* paused conversation, and using state history for time-travel
debugging.


In [12]:
# --- Resuming a paused conversation in a fresh call, by thread_id only ------
# (Simulates coming back to this later / a different part of the app resuming
# the same session -- no need to hold onto the original state object.)

resume_config = {"configurable": {"thread_id": "client-session-1"}}
state_snapshot = full_graph.get_state(resume_config)
print("Resumed state values:", state_snapshot.values)
print("Next node(s) to run:", state_snapshot.next)


Resumed state values: {'question': 'Which laptop should I recommend to a budget-conscious client?', 'target_products': ['laptop a', 'laptop b', 'laptop c'], 'retrieved_data': 'laptop a: $799 (8GB RAM, 256GB SSD)\nlaptop b: $1199 (16GB RAM, 512GB SSD)\nlaptop c: $549 (8GB RAM, 128GB SSD)', 'draft': 'For a budget-conscious client, I would recommend Laptop C, which offers a great balance of features and affordability at $549. With 8GB RAM and a 128GB SSD, it provides sufficient performance for everyday tasks while being the most cost-effective option. This choice allows your client to stay within their budget while still getting a reliable laptop for basic use.', 'critique': 'The recommendation is clear and names a specific product, "Laptop C", with a price, but it does not provide a well-known brand name for the laptop.', 'quality_score': 0.9, 'revision_count': 1, 'max_revisions': 2, 'approved': True, 'final_answer': '[SENT TO CLIENT] Recommendation:\nFor a budget-conscious client, I wou

In [13]:
# --- Time travel: inspect every checkpoint recorded for this thread --------
history = list(full_graph.get_state_history(config))
print(f"{len(history)} checkpoints recorded for this run.\n")

for i, snap in enumerate(reversed(history)):
    node_reached = snap.metadata.get("step")
    print(f"checkpoint {i}: step={node_reached}, next={snap.next}, "
          f"revision_count={snap.values.get('revision_count')}")


9 checkpoints recorded for this run.

checkpoint 0: step=-1, next=('__start__',), revision_count=None
checkpoint 1: step=0, next=('plan',), revision_count=0
checkpoint 2: step=1, next=('retrieve',), revision_count=0
checkpoint 3: step=2, next=('generate',), revision_count=0
checkpoint 4: step=3, next=('critique',), revision_count=0
checkpoint 5: step=4, next=('publish_gate',), revision_count=1
checkpoint 6: step=5, next=('publish_gate',), revision_count=1
checkpoint 7: step=6, next=('publish',), revision_count=1
checkpoint 8: step=7, next=(), revision_count=1


In [14]:
# --- Replay from an earlier checkpoint (e.g. re-run from right after 'generate') ---
# Pick a checkpoint whose `next` was ('critique',) -- i.e. right before critique ran --
# and re-invoke the graph from there. This re-runs everything downstream of that
# checkpoint, which is exactly what you want when debugging "what would have
# happened if critique/publish had behaved differently."
target_checkpoint = next(
    (s for s in history if s.next == ("critique",)), None
)
if target_checkpoint:
    replay_config = target_checkpoint.config
    print("Replaying from checkpoint before 'critique'...")
    for step in full_graph.stream(None, config=replay_config):
        print(step)
else:
    print("No matching checkpoint found in this run (depends on how many revisions happened).")


Replaying from checkpoint before 'critique'...
[CRITIQUE] pass 0: score=0.9, feedback='The recommendation is clear and names a specific product, "Laptop C", with a price, but it does not provide a well-known brand name for the laptop.'
[ROUTE] score 0.9 acceptable or out of retries -> publish_gate
{'critique': {'quality_score': 0.9, 'critique': 'The recommendation is clear and names a specific product, "Laptop C", with a price, but it does not provide a well-known brand name for the laptop.', 'revision_count': 1}}
{'__interrupt__': ()}


### LangChain `AgentExecutor` vs. LangGraph — when to reach for each

**`AgentExecutor`** is the right tool when the task really is "one loop, pick a tool, stop
when done" — a single agent with tools and maybe memory, no branching, no need to pause
mid-task, no multi-step workflow with named stages. It's less code to set up and easier to
reason about for that narrow shape.

**LangGraph** earns its extra setup cost as soon as the workflow needs any of: distinct
named stages with different logic (plan/retrieve/generate/critique here), a loop-back /
self-correction pattern, a pause point for human approval, persistence of state across
separate invocations (a paused run resumed later, possibly by a different process), or
replay/time-travel debugging. In short: reach for `AgentExecutor` for a single autonomous
tool-using agent; reach for LangGraph the moment "workflow" or "state machine" is a more
accurate description of the task than "agent loop."
